# Chai-1: ToxT + native El Tor *ctxAB* promoter

Runs Chai-1 on the **verified native El Tor promoter sequence** (positions -76 to -41),
as a cross-check on the AlphaFold3 model in Supplementary Section S.13.

**Before running:** Runtime -> Change runtime type -> **GPU** (A100 strongly preferred;
Chai-1 OOMs on <10 GB cards at default settings).

Sequences below were generated directly from the project's verified FASTA files
(`chai_run/toxt_dna_eltor.fasta`), not retyped. Cell 3 re-verifies them before inference.


## 1. Install Chai-1

In [ ]:
!pip install -q chai_lab
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('GPU:', p.name, '| VRAM: %.1f GB' % (p.total_memory/1024**3))
    if p.total_memory/1024**3 < 10:
        print('WARNING: <10 GB VRAM -- Chai-1 is likely to OOM. Switch to an A100 runtime.')

## 2. Write the input FASTA (1 protein copy)

In [ ]:
fasta = '''>protein|name=toxt
MIGKKSFQTNVYRMSKFDTYIFNNLYINDYKMFWIDSGIAKLIDKNCLVSYEINSSSIILLKKNSIQRFSLTSLSDENINVSVITISDSFIRSLKSYILGDLMIRNLYSENKDLLLWNCEHNDIAVLSEVVNGFREINYSDEFLKVFFSGFFSKVEKKYNSIFITDDLDAMEKISCLVKSDITRNWRWADICGELRTNRMILKKELESRGVKFRELINSIRISYSISLMKTGEFKIKQIAYQSGFASVSYFSTVFKSTMNVAPSEYLFMLTGVAEK
>dna|name=ctxAB_eltor_top
TTTTGATTTTTGATTTTTGATTTCAAATAATACAAA
>dna|name=ctxAB_eltor_bottom
TTTGTATTATTTGAAATCAAAAATCAAAAATCAAAA
'''

with open('toxt_dna_eltor.fasta','w') as f:
    f.write(fasta)
print(fasta)

## 3. Verify sequences BEFORE running (do not skip)

In [ ]:
def read_fasta(p):
    entries, name, seq = [], None, []
    for line in open(p):
        line = line.strip()
        if line.startswith('>'):
            if name: entries.append((name, ''.join(seq)))
            name, seq = line[1:], []
        elif line: seq.append(line)
    if name: entries.append((name, ''.join(seq)))
    return entries

e = read_fasta('toxt_dna_eltor.fasta')
prot = [s for n,s in e if n.startswith('protein')]
dna  = [s for n,s in e if n.startswith('dna')]
comp = {'A':'T','T':'A','G':'C','C':'G'}

assert len(prot)==1 and len(prot[0])==276, 'protein wrong'
assert len(dna)==2 and all(len(d)==36 for d in dna), 'dna length wrong'
assert dna[1]==''.join(comp[b] for b in reversed(dna[0])), 'strands are not reverse complements'
assert 'ATTTCAAAT' in dna[0], 'paper landmark ATTTCAAAT missing'
assert 'GCACATTTTAATAAAAT' not in dna[0], 'PLACEHOLDER SEQUENCE DETECTED -- STOP'
print('protein len :', len(prot[0]))
print('dna top     :', dna[0])
print('dna bottom  :', dna[1])
print('
ALL SEQUENCE CHECKS PASSED - safe to run inference')

## 4. Run Chai-1

In [ ]:
from pathlib import Path
from chai_lab.chai1 import run_inference

candidates = run_inference(
    fasta_file=Path('toxt_dna_eltor.fasta'),
    output_dir=Path('output_eltor'),
    use_esm_embeddings=True,
    use_msa_server=True,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    num_diffn_samples=5,
    seed=42,
    low_memory=True,
)
print('Done.')
print('Ranked scores:', candidates.ranking_data)

### If it OOMs
Re-run cell 4 with reduced settings (note these in the write-up, since they differ from defaults):
```python
num_trunk_recycles=1, num_diffn_timesteps=100, num_diffn_samples=2
```

## 5. Package results for download

In [ ]:
import shutil
shutil.make_archive('chai_eltor_results','zip','output_eltor')
from google.colab import files
files.download('chai_eltor_results.zip')

## Optional: 2-copy (dimer stoichiometry) run

Tests whether two ToxT copies engage the two toxboxes better than one -- the
EMSA-motivated hypothesis from Dittmer & Withey (2012). Run only if the 1-copy
job succeeded comfortably; it needs more memory.

In [ ]:
fasta2 = '''>protein|name=toxt_chainA
MIGKKSFQTNVYRMSKFDTYIFNNLYINDYKMFWIDSGIAKLIDKNCLVSYEINSSSIILLKKNSIQRFSLTSLSDENINVSVITISDSFIRSLKSYILGDLMIRNLYSENKDLLLWNCEHNDIAVLSEVVNGFREINYSDEFLKVFFSGFFSKVEKKYNSIFITDDLDAMEKISCLVKSDITRNWRWADICGELRTNRMILKKELESRGVKFRELINSIRISYSISLMKTGEFKIKQIAYQSGFASVSYFSTVFKSTMNVAPSEYLFMLTGVAEK
>protein|name=toxt_chainB
MIGKKSFQTNVYRMSKFDTYIFNNLYINDYKMFWIDSGIAKLIDKNCLVSYEINSSSIILLKKNSIQRFSLTSLSDENINVSVITISDSFIRSLKSYILGDLMIRNLYSENKDLLLWNCEHNDIAVLSEVVNGFREINYSDEFLKVFFSGFFSKVEKKYNSIFITDDLDAMEKISCLVKSDITRNWRWADICGELRTNRMILKKELESRGVKFRELINSIRISYSISLMKTGEFKIKQIAYQSGFASVSYFSTVFKSTMNVAPSEYLFMLTGVAEK
>dna|name=ctxAB_eltor_top
TTTTGATTTTTGATTTTTGATTTCAAATAATACAAA
>dna|name=ctxAB_eltor_bottom
TTTGTATTATTTGAAATCAAAAATCAAAAATCAAAA
'''
with open('toxt_dna_eltor_2copy.fasta','w') as f:
    f.write(fasta2)

from pathlib import Path
from chai_lab.chai1 import run_inference
candidates2 = run_inference(
    fasta_file=Path('toxt_dna_eltor_2copy.fasta'),
    output_dir=Path('output_eltor_2copy'),
    use_esm_embeddings=True, use_msa_server=True,
    num_trunk_recycles=3, num_diffn_timesteps=200, num_diffn_samples=5,
    seed=42, low_memory=True,
)
print('Ranked scores:', candidates2.ranking_data)
import shutil; shutil.make_archive('chai_eltor_2copy_results','zip','output_eltor_2copy')
from google.colab import files; files.download('chai_eltor_2copy_results.zip')